In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbw_orion_dev_uks_001.orion_bronze.raw_jpl_horizons (
  ingestion_run_id STRING,
  source_system STRING,
  source_endpoint STRING,
  request_url STRING,
  request_params_json STRING,
  response_status_code INT,
  response_body STRING,
  response_hash STRING,
  ingested_at TIMESTAMP,
  mission_name STRING,
  mission_start_date DATE,
  mission_end_date DATE
)
USING DELTA
COMMENT 'Raw JPL Horizons API responses for Project Orion';

CREATE TABLE IF NOT EXISTS dbw_orion_dev_uks_001.orion_bronze.raw_donki_events (
  ingestion_run_id STRING,
  source_system STRING,
  source_endpoint STRING,
  request_url STRING,
  request_params_json STRING,
  response_status_code INT,
  response_body STRING,
  response_hash STRING,
  ingested_at TIMESTAMP,
  mission_name STRING,
  mission_start_date DATE,
  mission_end_date DATE
)
USING DELTA
COMMENT 'Raw NASA DONKI API responses for Project Orion';

CREATE TABLE IF NOT EXISTS dbw_orion_dev_uks_001.orion_audit.ingestion_runs (
  ingestion_run_id STRING,
  pipeline_name STRING,
  source_system STRING,
  run_started_at TIMESTAMP,
  run_finished_at TIMESTAMP,
  status STRING,
  records_written BIGINT,
  error_message STRING
)
USING DELTA
COMMENT 'Audit table for Project Orion ingestion runs';

# Bronze Ingestion — JPL Horizons

Ingests raw orbital ephemeris response data from NASA/JPL Horizons into the Project Orion Bronze layer.

This notebook intentionally stores the raw response body unchanged. Parsing and normalization happen later in Silver.

In [0]:
import hashlib
import json
import uuid
from datetime import datetime, timezone

import requests
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType,
    DateType,
)

In [0]:
CATALOG = "dbw_orion_dev_uks_001"

BRONZE_JPL_TABLE = f"{CATALOG}.orion_bronze.raw_jpl_horizons"
AUDIT_TABLE = f"{CATALOG}.orion_audit.ingestion_runs"

MISSION_NAME = "artemis_ii"
MISSION_START_DATE = "2026-04-01"
MISSION_END_DATE = "2026-04-10"

JPL_HORIZONS_BASE_URL = "https://ssd.jpl.nasa.gov/api/horizons.api"

In [0]:
request_params = {
    "format": "json",
    "COMMAND": "301",              # Moon
    "OBJ_DATA": "YES",
    "MAKE_EPHEM": "YES",
    "EPHEM_TYPE": "VECTORS",
    "CENTER": "500@399",           # Geocentric / Earth center
    "START_TIME": MISSION_START_DATE,
    "STOP_TIME": MISSION_END_DATE,
    "STEP_SIZE": "1 h",
    "VEC_TABLE": "2",
    "CSV_FORMAT": "YES",
}

COMMAND = 301      Moon
CENTER = 500@399   Earth center
EPHEM_TYPE=VECTORS vector-style ephemeris output
STEP_SIZE=1 hour   enough data to prove distributed processing later

In [0]:
ingestion_run_id = str(uuid.uuid4())
run_started_at = datetime.now(timezone.utc)

response = requests.get(
    JPL_HORIZONS_BASE_URL,
    params=request_params,
    timeout=60,
)

run_finished_at = datetime.now(timezone.utc)

response_body = response.text
response_hash = hashlib.sha256(response_body.encode("utf-8")).hexdigest()

print(f"Run ID: {ingestion_run_id}")
print(f"Status code: {response.status_code}")
print(f"Response length: {len(response_body):,} characters")
print(f"Response hash: {response_hash}")
print(response.url)

In [0]:
bronze_schema = StructType([
    StructField("ingestion_run_id", StringType(), False),
    StructField("source_system", StringType(), False),
    StructField("source_endpoint", StringType(), False),
    StructField("request_url", StringType(), False),
    StructField("request_params_json", StringType(), False),
    StructField("response_status_code", IntegerType(), False),
    StructField("response_body", StringType(), True),
    StructField("response_hash", StringType(), False),
    StructField("ingested_at", TimestampType(), False),
    StructField("ingested_date", DateType(), False),
    StructField("mission_name", StringType(), False),
    StructField("mission_start_date", DateType(), False),
    StructField("mission_end_date", DateType(), False),
])

In [0]:
bronze_df.createOrReplaceTempView("stg_raw_jpl_horizons")

spark.sql(f"""
INSERT INTO {BRONZE_JPL_TABLE} (
  ingestion_run_id,
  source_system,
  source_endpoint,
  request_url,
  request_params_json,
  response_status_code,
  response_body,
  response_hash,
  ingested_at,
  ingested_date,
  mission_name,
  mission_start_date,
  mission_end_date
)
SELECT
  ingestion_run_id,
  source_system,
  source_endpoint,
  request_url,
  request_params_json,
  response_status_code,
  response_body,
  response_hash,
  ingested_at,
  ingested_date,
  mission_name,
  mission_start_date,
  mission_end_date
FROM stg_raw_jpl_horizons
""")

In [0]:
display(
    spark.sql(f"""
        SELECT
            ingestion_run_id,
            source_system,
            source_endpoint,
            response_status_code,
            length(response_body) AS response_body_length,
            response_hash,
            ingested_at,
            ingested_date,
            mission_name
        FROM {BRONZE_JPL_TABLE}
        ORDER BY ingested_at DESC
        LIMIT 10
    """)
)

In [0]:
print(response_body)